- notebook 侧记得把这一行加进配置 cell（kernel 的子进程不会读你的 .bashrc）：

In [ ]:
import os
for k, v in {
    "UV_CACHE_DIR":     "/data2/c220/hz/.uv-cache",
    "TRITON_CACHE_DIR": "/data2/c220/hz/.triton-cache",
    "HF_HOME":          "/data2/c220/hz/.hf-cache",
}.items():
    os.environ[k] = v


In [1]:
import os, shutil

REPO    = "/data2/c220/hz/cs336"
SUBTREE = f"{REPO}/third_party/assignment2-systems"

# ---- 1. 目录骨架 ----
os.makedirs(f"{REPO}/cs336_systems", exist_ok=True)
for f in [f"{REPO}/cs336_systems/__init__.py", f"{REPO}/README.md"]:
    open(f, "a").close()   # 不存在则创建，存在则不动

# ---- 2. 复制官方 tests（已存在则跳过，保护你填过的 adapters）----
if not os.path.exists(f"{REPO}/tests"):
    shutil.copytree(f"{SUBTREE}/tests", f"{REPO}/tests")
    print("✅ tests/ copied from subtree")
else:
    print("ℹ️  tests/ 已存在，跳过")

# ---- 3. 写 pyproject.toml（FORCE=True 可覆盖重写）----
PYPROJECT = '''
[project]
name = "cs336-assignment2"
version = "0.1.0"
description = "My CS336 Assignment 2 - Systems"
readme = "README.md"
requires-python = ">=3.12,<3.14"
dependencies = [
    "cs336-basics",
    "humanfriendly",
    "matplotlib",
    "numpy",
    "pytest>=8",
    "regex>=2026.3.32",
    "torch~=2.11.0",
    "tqdm>=4.67.1",
    "wandb>=0.19.7",
    "pandas>=2",
    "ty>=0.0.29",
    "ruff>=0.15.10",
]

[build-system]
requires = ["uv_build>=0.11.6,<0.12"]
build-backend = "uv_build"

[tool.uv.build-backend]
module-name = "cs336_systems"
module-root = ""

[tool.uv]
package = true
python-preference = "managed"

[tool.uv.sources]
cs336-basics = { path = "./third_party/assignment2-systems/cs336-basics", editable = true }

[tool.pytest.ini_options]
log_cli = true
log_cli_level = "WARNING"
addopts = "-s"
filterwarnings = [
    "ignore::DeprecationWarning:torch.jit.*",
    "ignore:Attempting to run cuBLAS:UserWarning",
    "ignore:TensorFloat32:UserWarning",
]
testpaths = ["tests"]
norecursedirs = ["third_party", ".venv", ".git", "build"]

[tool.ruff]
line-length = 180
[tool.ruff.lint.extend-per-file-ignores]
"__init__.py" = ["E402", "F401", "F403", "E501"]
[tool.ruff.lint]
extend-select = ["UP"]
ignore = ["F722"]
'''

FORCE = False   # ← 上次写坏过 pyproject 就改 True 重写
pp = f"{REPO}/pyproject.toml"
if FORCE or not os.path.exists(pp):
    open(pp, "w").write(PYPROJECT)
    print("✅ pyproject.toml written")
else:
    print("ℹ️  pyproject.toml 已存在，跳过（FORCE=True 可覆盖）")

# ---- 4. .gitignore ----
need = [".venv/", "__pycache__/", "*.egg-info/", "uv.lock.bak"]
gi = f"{REPO}/.gitignore"
existing = open(gi).read() if os.path.exists(gi) else ""
with open(gi, "a") as f:
    for line in need:
        if line not in existing:
            f.write(f"\n{line}")
print("✅ .gitignore ok")

# ---- 5. 自检报告 ----
print("─" * 60)
for name, p in {
    "pyproject":  pp,
    "cs336_systems":  f"{REPO}/cs336_systems/__init__.py",
    "tests/adapters": f"{REPO}/tests/adapters.py",
    "README.md":  f"{REPO}/README.md",
    "uv.lock (sync后)": f"{REPO}/uv.lock",
    ".venv (sync后)":   f"{REPO}/.venv/bin/python",
}.items():
    print(f"{'✅' if os.path.exists(p) else '⬜'} {name}")


✅ tests/ copied from subtree
✅ pyproject.toml written
✅ .gitignore ok
────────────────────────────────────────────────────────────
✅ pyproject
✅ cs336_systems
✅ tests/adapters
✅ README.md
⬜ uv.lock (sync后)
⬜ .venv (sync后)
